# EnMAP mineral-classification — dataset showcase (magnesite)

This notebook demonstrates what the pipeline produced for **one** EnMAP L2A scene clipped around a mine locality:

- Load `manifest.csv`
- Open one sample `.npz` (reflectance cube `X`, mask `y`, wavelengths)
- Visualize an RGB composite and the OSM mine/quarry mask

Outputs come from:

- Step 03: `data/magnesite/chips/.../*.nc`
- Step 04: `data/magnesite/masks/.../*_osm_mask.npy`
- Step 05: `data/magnesite/dataset/samples/*.npz`


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATASET_DIR = Path("data/magnesite/dataset")
MANIFEST = DATASET_DIR / "manifest.csv"

assert MANIFEST.exists(), f"Missing {MANIFEST} — run step 05 first"
manifest = pd.read_csv(MANIFEST)
manifest.head(3)


In [ ]:
# Pick one sample (first row)
row = manifest.iloc[0].to_dict()
npz_path = Path(row["npz_path"])  # stored relative to repo root
assert npz_path.exists(), f"Missing sample: {npz_path}"

sample = np.load(npz_path)

X = sample["X"]          # (B, H, W) float32 reflectance
y = sample["y"]          # (H, W) uint8 mask (0/1)
wl = sample["wavelength"] # (B,) float32 band centers (nm)

chip_id = str(sample["chip_id"])
mine_name = str(sample["mine_name"])
scene_id = str(sample["scene_id"])

print("chip_id:", chip_id)
print("mine_name:", mine_name)
print("scene_id:", scene_id)
print("X shape (B,H,W):", X.shape, X.dtype)
print("y shape (H,W):", y.shape, y.dtype, "positive pixels:", int(y.sum()))
print("wavelengths:", float(wl.min()), "..", float(wl.max()), "nm (B=", len(wl), ")")


In [ ]:
def _nearest_band_index(wavelengths_nm: np.ndarray, target_nm: float) -> int:
    return int(np.argmin(np.abs(wavelengths_nm - target_nm)))

def make_rgb(X_bhw: np.ndarray, wl_nm: np.ndarray, r_nm=650.0, g_nm=560.0, b_nm=490.0) -> np.ndarray:
    r_i = _nearest_band_index(wl_nm, r_nm)
    g_i = _nearest_band_index(wl_nm, g_nm)
    b_i = _nearest_band_index(wl_nm, b_nm)

    rgb = np.stack([X_bhw[r_i], X_bhw[g_i], X_bhw[b_i]], axis=-1)

    # robust stretch (per-channel)
    lo = np.nanpercentile(rgb, 2, axis=(0, 1), keepdims=True)
    hi = np.nanpercentile(rgb, 98, axis=(0, 1), keepdims=True)
    rgb = (rgb - lo) / np.maximum(hi - lo, 1e-6)
    return np.clip(rgb, 0, 1)

rgb = make_rgb(X, wl)

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(rgb)
ax[0].set_title("RGB composite")
ax[0].axis("off")

ax[1].imshow(y, cmap="gray")
ax[1].set_title("OSM mask (y)")
ax[1].axis("off")

ax[2].imshow(rgb)
ax[2].imshow(y, cmap="Reds", alpha=0.35)
ax[2].set_title("Overlay")
ax[2].axis("off")

plt.suptitle(f"{mine_name} — {scene_id}")
plt.tight_layout()
plt.show()
